In [4]:
import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

train_dir = 'split_data/train'
val_dir = 'split_data/val'

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

image_datasets = {
    'train': datasets.ImageFolder(train_dir, transform=data_transforms['train']),
    'val': datasets.ImageFolder(val_dir, transform=data_transforms['val'])
}

batch_size = 32
dataloaders = {
    'train': DataLoader(image_datasets['train'], batch_size=batch_size, shuffle=True, num_workers=2 if os.name != 'nt' else 0),
    'val': DataLoader(image_datasets['val'], batch_size=batch_size, shuffle=False, num_workers=2 if os.name != 'nt' else 0)
}

print(f"Training batches: {len(dataloaders['train'])} (Total images: {len(image_datasets['train'])})")
print(f"Validation batches: {len(dataloaders['val'])} (Total images: {len(image_datasets['val'])})")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Training batches: 1188 (Total images: 37997)
Validation batches: 255 (Total images: 8129)
Using device: cpu


In [5]:
import torch.nn as nn
from torchvision import models

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features

num_classes = len(image_datasets['train'].classes)

model.fc = nn.Linear(num_features, num_classes)

model = model.to(device)

print(f"Model successfully loaded and configured!")
print(f"Input features to final layer: {num_features}")
print(f"Output classes (Your crop diseases): {num_classes}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Anmol Trivedi/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


Model successfully loaded and configured!
Input features to final layer: 512
Output classes (Your crop diseases): 38


In [6]:
import torch.optim as optim
import time
import copy

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

print("Loss function and Optimizer initialized successfully!")

Loss function and Optimizer initialized successfully!


In [7]:
def train_model(model, criterion, optimizer, num_epochs=3):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluation mode

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # Statistics tracking
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(image_datasets[phase])
            epoch_acc = running_corrects.double() / len(image_datasets[phase])

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

    time_elapsed = time.time() - since
    print(f'\nTraining complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best Val Acc: {best_acc:4f}')

    model.load_state_dict(best_model_wts)
    return model

In [8]:
# Train the model for 3 epochs
trained_model = train_model(model, criterion, optimizer, num_epochs=3)

os.makedirs('models', exist_ok=True)
torch.save(trained_model.state_dict(), 'models/crop_disease_resnet18.pth')
print("Model weights successfully saved to models/crop_disease_resnet18.pth!")


Epoch 1/3
----------
Train Loss: 0.7072 Acc: 0.8341
Val Loss: 0.2672 Acc: 0.9269

Epoch 2/3
----------
Train Loss: 0.2764 Acc: 0.9231
Val Loss: 0.2262 Acc: 0.9316

Epoch 3/3
----------
Train Loss: 0.2247 Acc: 0.9317
Val Loss: 0.1831 Acc: 0.9419

Training complete in 68m 52s
Best Val Acc: 0.941936
Model weights successfully saved to models/crop_disease_resnet18.pth!


In [9]:
with open('models/classes.txt', 'w') as f:
    for class_name in image_datasets['train'].classes:
        f.write(f"{class_name}\n")
print("Class labels saved to models/classes.txt!")

Class labels saved to models/classes.txt!
